In [1]:
from model_training import *
from model_training import _new_pipeline

In [2]:
df_final = load_training_data()
test = load_test_data()
print(f"Train: {len(df_final):,} rows")
print(df_final['sentiment'].value_counts())
print(f"\nTest: {len(test):,} rows")
print(test['sentiment'].value_counts())
print(f"\nTest texts also in train: {test['text'].isin(df_final['text']).sum()}")

Train: 45,633 rows
sentiment
neutral     20673
positive    17853
negative     7107
Name: count, dtype: int64

Test: 1,000 rows
sentiment
neutral     483
negative    324
positive    193
Name: count, dtype: int64

Test texts also in train: 0


In [3]:
model, vectorizer, data_count, metrics = train_model()

print(f"Test set         : {metrics['test_size']:,} rows")
print(f"Accuracy         : {metrics['accuracy']:.4f}")
print(f"Precision (macro): {metrics['precision_macro']:.4f}")
print(f"Recall (macro)   : {metrics['recall_macro']:.4f}")
print(f"F1 (macro)       : {metrics['f1_macro']:.4f}")
print(f"Latency          : {metrics['latency_ms']:.3f} ms/sentence")

Test set         : 1,000 rows
Accuracy         : 0.6110
Precision (macro): 0.5973
Recall (macro)   : 0.6128
F1 (macro)       : 0.6036
Latency          : 0.651 ms/sentence


In [4]:
print("Per-class report:")
print(metrics['report'].round(3))

Per-class report:
              precision  recall  f1-score  support
negative          0.613   0.627     0.620    324.0
neutral           0.652   0.600     0.625    483.0
positive          0.527   0.611     0.566    193.0
macro avg         0.597   0.613     0.604   1000.0
weighted avg      0.615   0.611     0.612   1000.0


In [5]:
print("Confusion matrix:")
print(metrics['confusion_matrix'])

Confusion matrix:
                 pred_negative  pred_neutral  pred_positive
actual_negative            203            97             24
actual_neutral             111           290             82
actual_positive             17            58            118


In [6]:
import time
from sklearn.pipeline import FeatureUnion

X_tr, y_tr = df_final['text_clean'], df_final['sentiment']
X_te, y_te = test['text'].apply(clean_for_rules), test['sentiment']

def word_only():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)

def word_char():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])

for name, make_vec in [("word (1,2)", word_only), ("word + char (2,5)", word_char)]:
    start = time.time()
    vec = make_vec()
    clf = LinearSVC(C=0.3, class_weight='balanced', random_state=42, max_iter=5000)
    clf.fit(vec.fit_transform(X_tr), y_tr)
    pred = clf.predict(vec.transform(X_te))
    print(f"{name:18s} | Accuracy: {accuracy_score(y_te, pred):.4f} "
          f"| F1-macro: {precision_recall_fscore_support(y_te, pred, average='macro')[2]:.4f} "
          f"| Features: {len(vec.get_feature_names_out()):,} | Time: {time.time() - start:.1f}s")

word (1,2)         | Accuracy: 0.6110 | F1-macro: 0.6036 | Features: 90,281 | Time: 3.1s
word + char (2,5)  | Accuracy: 0.6150 | F1-macro: 0.6148 | Features: 177,040 | Time: 17.9s


In [7]:
err = pd.DataFrame({"text": test["text"], "actual": test["sentiment"], "pred": metrics["predictions"]})
err = err[(err.actual != err.pred) & (err.actual != "neutral") & (err.pred != "neutral")]
print(f"{len(err):,} negative <-> positive errors\n")
for _, r in err.sample(min(20, len(err)), random_state=0).iterrows():
    print(f"[{r.actual} -> {r.pred}] {str(r.text)[:150]}")

41 negative <-> positive errors

[positive -> negative] @user @user @user @user @user I teach many kids healthy because of vaccines...
[positive -> negative] ... well that was fucking beautiful<3  #westworld
[negative -> positive] The Socialist/Fascist Party (aka. Dems) is driving itself into obscurity.
[positive -> negative] My idols are #littlemix #justinbieber #arianagrande
[negative -> positive] @user @user Gotta love experts on Putin style.
[negative -> positive] I've interacted with liberals that are woke af until they start arguing that drone strikes are actually good for feminism.. or something.
[negative -> positive] WAIT! COME BACK! NOOOO! [burp] - Mike Pence as
[positive -> negative] The movie where Zac Efron is a DJ is super fucking rad
[negative -> positive] @user although it'd be extra nice is Maduro (president of Venezuela, where i live) could be included in that pack with Fidel
[positive -> negative] I can't believe this is happening. #ThankYouObama
[positive -> negativ